In [ ]:
import math
import json
import threading
from urllib.request import Request, urlopen
from urllib.parse import urlencode
import webbrowser
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import date
import numpy as np
import pandas as pd
import pvlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Catanauan, Quezon, Philippines; user-supplied site coordinates
DEFAULT_LAT, DEFAULT_LON = 13.647841, 122.279707
TIMEZONE = 'Asia/Manila'
PANELS_PER_ARRAY, PANEL_WATTS = 20, 500
ARRAY_KWP = PANELS_PER_ARRAY * PANEL_WATTS / 1000
DT = 5 / 60  # hours per five-minute step
INVERTER_EFF, CHARGE_EFF, DISCHARGE_EFF = 0.94, 0.95, 0.95
DOD, AUTONOMY_HOURS = 0.80, 48
INVERTER_LIMIT_KW = 20.0  # illustrative shared inverter capacity; configurable in real design
FIXED_TILT, FIXED_AZIMUTH = 15.0, 180.0  # true south


def load_profile(hours):
    """Illustrative 2–4 kW AC demand, repeated daily."""
    h = np.asarray(hours) % 24
    p = np.full(h.shape, 2.0)
    p[(h >= 6) & (h < 9)] = 2.5
    p[(h >= 9) & (h < 15)] = 3.0
    p[(h >= 15) & (h < 18)] = 3.5
    p[(h >= 18) & (h < 22)] = 4.0
    return p


def panel_normal(tilt, azimuth):
    t, a = math.radians(tilt), math.radians(azimuth)
    return np.array([math.sin(t)*math.sin(a), math.sin(t)*math.cos(a), math.cos(t)])


def panel_corners(center, normal):
    right = np.cross([0., 0., 1.], normal)
    if np.linalg.norm(right) < 1e-8:
        right = np.array([1., 0., 0.])
    right /= np.linalg.norm(right)
    up = np.cross(normal, right)
    up /= np.linalg.norm(up)
    c = np.asarray(center)
    width, height = 2.3, 2.1
    return [c-right*width/2-up*height/2,
            c+right*width/2-up*height/2,
            c+right*width/2+up*height/2,
            c-right*width/2+up*height/2]


def pv_power(poa, elevation):
    irr = np.maximum(0, np.asarray(poa, dtype=float))
    cell_temp = 30 + (25 / 800) * irr
    temperature_factor = np.maximum(0, 1 - 0.0035*(cell_temp-25))
    power = ARRAY_KWP * irr/1000 * temperature_factor * 0.96
    power = np.clip(power, 0, ARRAY_KWP)
    power[np.asarray(elevation) <= 0] = 0
    return power


def calculate_arrays(day, lat, lon):
    times = pd.date_range(pd.Timestamp(day, tz=TIMEZONE), periods=288, freq='5min')
    loc = pvlib.location.Location(lat, lon, tz=TIMEZONE)
    position = loc.get_solarposition(times)
    sky = loc.get_clearsky(times, model='ineichen')
    zen = position['apparent_zenith'].to_numpy()
    elevation = position['apparent_elevation'].to_numpy()
    sun_az = position['azimuth'].to_numpy()
    tracking_tilt = np.where(elevation > 0, np.clip(zen, 0, 85), 15.)
    tracking_az = np.where(elevation > 0, np.clip(sun_az, 60, 300), 180.)

    def irradiance(tilt, az):
        return pvlib.irradiance.get_total_irradiance(
            surface_tilt=tilt, surface_azimuth=az,
            solar_zenith=zen, solar_azimuth=sun_az,
            dni=sky['dni'].to_numpy(), ghi=sky['ghi'].to_numpy(),
            dhi=sky['dhi'].to_numpy(), albedo=0.2)['poa_global']

    tracking_kw = pv_power(irradiance(tracking_tilt, tracking_az), elevation)
    fixed_kw = pv_power(irradiance(FIXED_TILT, FIXED_AZIMUTH), elevation)
    return dict(times=times, elevation=elevation, sun_az=sun_az,
                tracking_tilt=tracking_tilt, tracking_az=tracking_az,
                tracking_kw=tracking_kw, fixed_kw=fixed_kw)


def simulate_battery(pv_dc, load_ac, battery_kwh):
    reserve = battery_kwh * (1 - DOD)
    energy = battery_kwh  # fully charged at start of day
    soc, battery_kw, unmet, curtailed = [], [], [], []
    for pv, load in zip(pv_dc, load_ac):
        # Approximate DC-bus energy balance. Load uses inverter conversion efficiency.
        required_dc_kw = min(load, INVERTER_LIMIT_KW) / INVERTER_EFF
        extra_unmet = max(0, load-INVERTER_LIMIT_KW)*DT
        surplus = max(0, pv-required_dc_kw)
        deficit = max(0, required_dc_kw-pv)
        if surplus > 0:
            potential = surplus*DT*CHARGE_EFF
            accepted = min(potential, battery_kwh-energy)
            energy += accepted
            battery_kw.append(accepted/DT)  # positive: stored energy rate
            curtailed.append(max(0, surplus*DT-accepted/CHARGE_EFF))
            unmet.append(extra_unmet)
        else:
            need = deficit*DT/DISCHARGE_EFF
            taken = min(need, max(0, energy-reserve))
            energy -= taken
            battery_kw.append(-taken/DT)  # negative: energy withdrawal rate
            curtailed.append(0.)
            unmet.append(extra_unmet+max(0, deficit*DT-taken*DISCHARGE_EFF)*INVERTER_EFF)
        soc.append(100*energy/battery_kwh)
    return dict(soc=np.array(soc), battery_kw=np.array(battery_kw),
                unmet=np.array(unmet), curtailed=np.array(curtailed))


class SolarGUI:
    def __init__(self, root):
        self.root = root
        root.title('SOLAR SIMULATOR V6 | TRACKING + SOUTH-FACING FIXED')
        root.geometry('1550x950')
        self.lat = tk.StringVar(value=str(DEFAULT_LAT))
        self.lon = tk.StringVar(value=str(DEFAULT_LON))
        self.day = tk.StringVar(value=date.today().isoformat())
        self.battery_override = tk.StringVar(value='0')
        self.supply = tk.StringVar(value='Both arrays (20 kWp)')
        self.view = tk.StringVar(value='Both')
        self.hour = tk.DoubleVar(value=8)
        self.running = False
        self.place_name = 'Location lookup pending'
        self.applied_coords = None
        self.lookup_serial = 0
        self.lookup_job = None
        self.data = None
        self.build_ui()
        self.recalculate()
        self.lookup_location()

    def build_ui(self):
        top = ttk.Frame(self.root, padding=7)
        top.pack(fill='x')
        for label, var, width in [('Latitude', self.lat, 11), ('Longitude', self.lon, 12),
                                  ('Date YYYY-MM-DD', self.day, 12),
                                  ('Battery kWh (0=Auto)', self.battery_override, 9)]:
            ttk.Label(top, text=label).pack(side='left', padx=(5, 2))
            ttk.Entry(top, textvariable=var, width=width).pack(side='left')
        ttk.Button(top, text='SET LOCATION', command=self.set_location).pack(side='left', padx=6)
        self.lat.trace_add('write', self.schedule_location_update)
        self.lon.trace_add('write', self.schedule_location_update)
        ttk.Button(top, text='CALCULATE', command=self.recalculate).pack(side='left', padx=4)
        ttk.Button(top, text='SATELLITE VIEW', command=self.satellite).pack(side='left', padx=4)
        second = ttk.Frame(self.root, padding=7)
        second.pack(fill='x')
        ttk.Label(second, text='Battery supplied by:').pack(side='left')
        ttk.Combobox(second, textvariable=self.supply, state='readonly', width=25,
                     values=['Tracking only (10 kWp)', 'Fixed south only (10 kWp)',
                             'Both arrays (20 kWp)']).pack(side='left', padx=5)
        ttk.Label(second, text='3D display:').pack(side='left', padx=(15, 2))
        ttk.Combobox(second, textvariable=self.view, state='readonly', width=13,
                     values=['Both', 'Tracking only', 'Fixed only']).pack(side='left', padx=5)
        ttk.Button(second, text='APPLY OPTIONS', command=self.recalculate).pack(side='left', padx=5)
        ttk.Button(second, text='PLAY / PAUSE', command=self.toggle).pack(side='left', padx=5)
        ttk.Label(second, text='Fixed array: 15° tilt, true south (180°)').pack(side='left', padx=12)
        self.location_label = ttk.Label(self.root, padding=5)
        self.location_label.pack(anchor='w')
        self.live_label = ttk.Label(self.root, padding=5)
        self.live_label.pack(anchor='w')
        ttk.Scale(self.root, from_=0, to=23.9167, variable=self.hour,
                  command=lambda _: self.draw()).pack(fill='x', padx=12)
        self.fig = Figure(figsize=(15, 8), dpi=100)
        self.ax3 = self.fig.add_subplot(221, projection='3d')
        self.axp = self.fig.add_subplot(222)
        self.axs = self.fig.add_subplot(223)
        self.axb = self.fig.add_subplot(224)
        self.fig.subplots_adjust(left=.07, right=.97, bottom=.10, top=.93,
                                 hspace=.39, wspace=.28)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        self.canvas.get_tk_widget().pack(fill='both', expand=True)
        self.status = ttk.Label(self.root, padding=7)
        self.status.pack(fill='x')
        self.battery_status = ttk.Label(self.root, padding=7, font=('Arial', 11, 'bold'))
        self.battery_status.pack(fill='x')
        ttk.Label(self.root, padding=7, text=(
            'Place names via OpenStreetMap Nominatim (internet required). Coordinates determine solar position. '
            'Asia/Manila timezone is fixed: use Philippine sites. Illustrative clear-sky model; no shading, '
            'clouds, motor use or detailed battery limits.')).pack(fill='x')

    def read_inputs(self):
        lat, lon = float(self.lat.get()), float(self.lon.get())
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError('Latitude must be -90..90; longitude -180..180.')
        d = date.fromisoformat(self.day.get())
        battery = float(self.battery_override.get())
        if battery < 0:
            raise ValueError('Battery override must be zero or positive.')
        return lat, lon, d, battery

    def schedule_location_update(self, *_):
        # Wait until the user stops typing before updating the project site.
        if self.lookup_job is not None:
            self.root.after_cancel(self.lookup_job)
        self.lookup_job = self.root.after(1100, self.set_location)

    def set_location(self):
        if self.lookup_job is not None:
            self.root.after_cancel(self.lookup_job)
            self.lookup_job = None
        self.recalculate()
        self.lookup_location()

    def lookup_location(self):
        # OpenStreetMap Nominatim: internet needed, approximate address labels.
        try:
            lat, lon = float(self.lat.get()), float(self.lon.get())
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                return
        except ValueError:
            return
        self.lookup_serial += 1
        serial = self.lookup_serial
        self.place_name = 'Looking up place name...'
        self.show_location_label()

        def worker():
            params = urlencode({'lat': lat, 'lon': lon, 'format': 'json',
                                'zoom': 14, 'addressdetails': 1})
            url = 'https://nominatim.openstreetmap.org/reverse?' + params
            try:
                request = Request(url, headers={
                    'User-Agent': 'SolarSimulatorEducational/1.0 (Python desktop demo)',
                    'Accept-Language': 'en'})
                with urlopen(request, timeout=9) as response:
                    data = json.load(response)
                a = data.get('address', {})
                parts = [a.get('city') or a.get('town') or a.get('municipality')
                         or a.get('village') or a.get('suburb') or a.get('county'),
                         a.get('state') or a.get('province'), a.get('country')]
                name = ', '.join(dict.fromkeys(x for x in parts if x))
                if not name:
                    name = data.get('display_name', 'No place name returned')
            except Exception:
                name = 'Place name unavailable (check internet); coordinates valid'

            def finish():
                if serial != self.lookup_serial:
                    return
                self.place_name = name
                self.show_location_label()
            try:
                self.root.after(0, finish)
            except RuntimeError:
                pass

        threading.Thread(target=worker, daemon=True).start()

    def show_location_label(self):
        if self.applied_coords is None:
            return
        lat, lon = self.applied_coords
        self.location_label.config(text=(
            f'PROJECT LOCATION: {self.place_name} | '
            f'Latitude {lat:.6f}, Longitude {lon:.6f} | '
            f'Simulation time: Philippine Time UTC+8 | Supply: {self.supply.get()}'))

    def satellite(self):
        try:
            lat, lon, _, _ = self.read_inputs()
            webbrowser.open(f'https://www.google.com/maps/@{lat},{lon},19z/data=!3m1!1e3')
        except Exception as e:
            messagebox.showerror('Location error', str(e))

    def recalculate(self):
        try:
            lat, lon, d, override = self.read_inputs()
            data = calculate_arrays(d, lat, lon)
            hours = np.arange(288)*DT
            load = load_profile(hours)
            supply = self.supply.get()
            if supply == 'Tracking only (10 kWp)':
                pv = data['tracking_kw']
            elif supply == 'Fixed south only (10 kWp)':
                pv = data['fixed_kw']
            else:
                pv = data['tracking_kw'] + data['fixed_kw']
            # Conservative 4 kW x 48h no-sun battery, accounting for conversion.
            auto_capacity = 4*AUTONOMY_HOURS/(DOD*INVERTER_EFF*DISCHARGE_EFF)
            capacity = override if override > 0 else auto_capacity
            battery = simulate_battery(pv, load, capacity)
            self.data, self.load, self.pv, self.battery = data, load, pv, battery
            self.capacity = capacity
            if self.applied_coords != (lat, lon):
                self.place_name = 'Location lookup pending'
            self.applied_coords = (lat, lon)
            self.show_location_label()
            self.draw()
        except Exception as e:
            messagebox.showerror('Calculation error', str(e))

    def toggle(self):
        self.running = not self.running
        if self.running:
            self.animate()

    def animate(self):
        if not self.running:
            return
        self.hour.set((self.hour.get()+DT) % 24)
        self.draw()
        self.root.after(120, self.animate)

    def draw_array(self, normal, offset_y, color):
        for row in range(4):
            for col in range(5):
                x = (col-2)*3.2
                y = (row-1.5)*3.9 + offset_y
                center = [x, y, 3]
                self.ax3.add_collection3d(Poly3DCollection(
                    [panel_corners(center, normal)], facecolors=color,
                    edgecolors='white', linewidths=.7))
                self.ax3.plot([x, x], [y, y], [0, 3], color='gray', lw=1.2)

    def draw(self):
        if self.data is None:
            return
        d, b = self.data, self.battery
        i = int(np.clip(round(self.hour.get()*12), 0, 287))
        hours = np.arange(288)*DT
        alt, az = float(d['elevation'][i]), float(d['sun_az'][i])
        tilt, face = float(d['tracking_tilt'][i]), float(d['tracking_az'][i])
        self.ax3.cla()
        view = self.view.get()
        if view in ('Both', 'Tracking only'):
            self.draw_array(panel_normal(tilt, face), -10 if view == 'Both' else 0, '#2178b5')
        if view in ('Both', 'Fixed only'):
            self.draw_array(panel_normal(FIXED_TILT, FIXED_AZIMUTH), 10 if view == 'Both' else 0, '#36a179')
        if alt > 0:
            sun_vec = panel_normal(90-alt, az)
            self.ax3.quiver(0, 0, 3.5, *(sun_vec*6), color='orange', linewidth=2.4,
                            arrow_length_ratio=.15)
        self.ax3.set(xlim=(-10, 10), ylim=(-21, 21) if view == 'Both' else (-11, 11),
                     zlim=(0, 10), xlabel='East (+x)', ylabel='North (+y)', zlabel='Height (m)')
        self.ax3.set_title('3D arrays: blue tracking | green fixed south')
        self.ax3.view_init(elev=25, azim=-60)

        self.axp.cla()
        self.axp.plot(hours, d['tracking_kw'], color='#2178b5', lw=2,
                      label=f'Tracking 10 kWp ({sum(d["tracking_kw"])*DT:.1f} kWh DC)')
        self.axp.plot(hours, d['fixed_kw'], color='#36a179', lw=2,
                      label=f'Fixed south 10 kWp ({sum(d["fixed_kw"])*DT:.1f} kWh DC)')
        self.axp.plot(hours, self.pv, color='orange', lw=1.7, linestyle='--', label='Selected PV supply')
        self.axp.step(hours, self.load, where='post', color='red', label='AC load 2–4 kW')
        self.axp.axvline(hours[i], color='gray', linestyle=':')
        self.axp.set(xlim=(0, 24), xlabel='Local time (PHT)', ylabel='Power (kW)',
                     title='Tracking vs fixed vs electrical load')
        self.axp.grid(alpha=.25)
        self.axp.legend(fontsize=7)

        self.axs.cla()
        self.axs.plot(hours, b['soc'], color='green', lw=2, label='Battery SOC')
        self.axs.axhline(20, color='red', linestyle='--', label='20% reserve')
        self.axs.axvline(hours[i], color='gray', linestyle=':')
        self.axs.scatter([hours[i]], [b['soc'][i]], color='black', zorder=5)
        self.axs.set(xlim=(0, 24), ylim=(0, 105), xlabel='Local time (PHT)',
                     ylabel='SOC (%)', title='Animated battery state of charge')
        self.axs.grid(alpha=.25)
        self.axs.legend(fontsize=8)

        self.axb.cla()
        self.axb.plot(hours, b['battery_kw'], color='blue', lw=2, label='Battery stored energy rate')
        self.axb.axhline(0, color='black', lw=.8)
        self.axb.axvline(hours[i], color='gray', linestyle=':')
        self.axb.scatter([hours[i]], [b['battery_kw'][i]], color='black', zorder=5)
        self.axb.set(xlim=(0, 24), xlabel='Local time (PHT)', ylabel='Battery kW',
                     title='Battery charge (+) / discharge (-)')
        self.axb.grid(alpha=.25)
        self.axb.legend(fontsize=8)

        self.live_label.config(text=(
            f'TIME: {d["times"][i].strftime("%Y-%m-%d %H:%M PHT")} | '
            f'Sun altitude {alt:.1f}° / azimuth {az:.1f}° | '
            f'Tracker tilt {tilt:.1f}° / azimuth {face:.1f}° | '
            f'Fixed tilt {FIXED_TILT:.0f}° / azimuth {FIXED_AZIMUTH:.0f}° (true south)'))
        self.status.config(text=(
            f'Tracking: {d["tracking_kw"][i]:.2f} kW DC | '
            f'Fixed south: {d["fixed_kw"][i]:.2f} kW DC | '
            f'Selected PV: {self.pv[i]:.2f} kW DC | '
            f'Load: {self.load[i]:.2f} kW AC | '
            f'Daily load: {sum(self.load)*DT:.1f} kWh AC | '
            f'Unmet: {sum(b["unmet"]):.2f} kWh AC'))
        bp = b['battery_kw'][i]
        state = 'CHARGING' if bp > .01 else ('DISCHARGING' if bp < -.01 else 'IDLE / FULL / RESERVE')
        self.battery_status.config(text=(
            f'BATTERY: {self.capacity:.1f} kWh nominal | '
            f'SOC: {b["soc"][i]:.1f}% | '
            f'Battery: {bp:+.2f} kW | {state} | '
            f'48-hour autonomy sized for continuous 4 kW load'))
        self.canvas.draw_idle()


if __name__ == '__main__':
    root = tk.Tk()
    app = SolarGUI(root)
    root.mainloop()
